Optimizing the previous NN

The problem with previous model (Training on GPU notebook)


When we run the model on the training data itself, the accuracy comes to around 98% which is a strong indicator of overfitting


Here's what can be done in order to reduce overfitting:

1. Adding more data - `We don't have enough data. We used all the data that was available`
2. Reducing the complexity of NN architecture - `This is already very simple`
3. Regularization - `We can do Regularization (simply adding a penalty term to loss function)`
4. Dropouts - ` We can apply this as well`
5. Data Augmentation - `Usually used with image data on CNN's `
6. Batch Normalization - `Can use`
7. Early Stopping - `Not to be covered in this NB`

Therefore, We will apply all these 3 techniques:

1. Regularization `Should be applied in optimizer code: weight_decay=0.0001`
2. Dropouts `Should be applied after activation function`
3. Batch Normalization `Should be applied before activation function`

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device:{device}")

Using Device:cuda
Using Device:cuda


In [3]:
df = pd.read_csv("fashion-mnist_train.csv")

In [4]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0
3,0,0,0,0,1,2,0,0,0,0,...,3.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,3,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
df.shape

(51553, 785)

In [6]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [7]:
import numpy as np
X = np.nan_to_num(X,nan=0.0)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [9]:
# performing the scailing
X_train = X_train/255.0
X_test = X_test/255.0

In [10]:
# Create a CustomDataset class
class CustomDataset(Dataset):

  def __init__(self, features, labels):

    self.features = torch.tensor(features, dtype=torch.float32)
    self.labels = torch.tensor(labels, dtype=torch.long)

  def __len__(self):

    return len(self.features)

  def __getitem__(self,index):

    return self.features[index], self.labels[index]

In [11]:
# create train_dataset object
train_dataset = CustomDataset(X_train, y_train)

In [12]:
test_dataset = CustomDataset(X_test,y_test)

In [13]:
# create train and test loader object
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True) # pin_memory is simply used to speed up the training on GPU
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True) # while predicting, you usually dont wanna shuffle the data.

In [17]:
# Define a NN class
class vrajNN(nn.Module):

  def __init__(self, num_features):

    super().__init__() # invoking the parent constructor

    self.model = nn.Sequential(
        nn.Linear(num_features, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(64, 10) # here we dont need to add softmax, because pytorch has internally made it by default.
    )

  def forward(self, X):

    return self.model(X)

In [18]:
# setting learning rate and epochs

epochs = 100
learning_rate = 0.1

In [19]:
# instantiate the model
model = vrajNN(X_train.shape[1]) # X_train.shape[1] is the no. of features
model.to(device)
# loss function
criterion = nn.CrossEntropyLoss()
# Optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

In [21]:
# training loop
for epoch in range(epochs):

  total_epoch_loss = 0 # setting up a variable in order to track the loss across each epoch

  for batch_features, batch_labels in train_loader:

    # move data to GPU
    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

    # forward pass
    outputs = model(batch_features)

    # loss calculate
    loss = criterion(outputs, batch_labels)

    # back propagation
    optimizer.zero_grad() # clearing out gradients
    loss.backward()

    # update weights
    optimizer.step()

    total_epoch_loss = total_epoch_loss + loss.item()

  avg_loss = total_epoch_loss / len(train_loader)
  print(f'Epoch: {epoch+1}, Loss : {avg_loss}')

Epoch: 1, Loss : 0.5085853717186545
Epoch: 2, Loss : 0.44288184723251084
Epoch: 3, Loss : 0.41473748549368145
Epoch: 4, Loss : 0.39045662236384776
Epoch: 5, Loss : 0.3735310808113141
Epoch: 6, Loss : 0.35875139045012466
Epoch: 7, Loss : 0.35001644924502673
Epoch: 8, Loss : 0.33854366284618087
Epoch: 9, Loss : 0.32777051872135227
Epoch: 10, Loss : 0.3196412125057588
Epoch: 11, Loss : 0.31424161569844666
Epoch: 12, Loss : 0.3044670908787715
Epoch: 13, Loss : 0.2989990040833308
Epoch: 14, Loss : 0.29324012502588626
Epoch: 15, Loss : 0.2884747524932894
Epoch: 16, Loss : 0.28263338498592006
Epoch: 17, Loss : 0.2777935597363533
Epoch: 18, Loss : 0.26960059893776317
Epoch: 19, Loss : 0.27190087362545884
Epoch: 20, Loss : 0.26625433579792435
Epoch: 21, Loss : 0.25990861217304195
Epoch: 22, Loss : 0.25555588842548077
Epoch: 23, Loss : 0.2560128270105811
Epoch: 24, Loss : 0.25109593428763843
Epoch: 25, Loss : 0.250008570280322
Epoch: 26, Loss : 0.24521216538014387
Epoch: 27, Loss : 0.24466155898

In [22]:
# set model to the evaluation mode
model.eval() # we have to explicitly tell the model that it's in eval mode because sometimes, some models might behave different while training and testing. for ex, dropouts.

vrajNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.2, inplace=False)
    (8): Linear(in_features=128, out_features=64, bias=True)
    (9): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.2, inplace=False)
    (12): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [23]:
# Evaluation code
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      # move data to GPU
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
        outputs = model(batch_features)

        _, predicted = torch.max(outputs, 1)  # Calculates class with max probability

        total += batch_labels.shape[0]      # Accumulate total samples

        correct += (predicted == batch_labels).sum().item()  # Count correct predictions

print(f"Accuracy: {correct / total:.4f}")  # or just correct/total

Accuracy: 0.8998


# Althought it might seem that the model's overall accuracy has not improved much, if we take a look at the previous notebook's case, the training data had about 98% accuracy, while this time, it's about 90 that signifies that overfitting has been reduced.